# 🛡️ Spam Detection QnA Bot — Lab 12
**Topic:** SMS Spam Detection (from Lab 10)

**Pipeline:**
1. Preprocess spam/ham dataset
2. Embed messages using Hugging Face MiniLM
3. Store vectors using FAISS
4. Search & match using similarity
5. Flask + HTML UI for interaction

## Step 0: Install Dependencies

In [ ]:
# Run this cell once to install required libraries
!pip install sentence-transformers faiss-cpu flask pandas numpy

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import faiss
from sentence_transformers import SentenceTransformer

print('✓ All libraries imported successfully!')

## Step 2: Load & Preprocess SMS Spam Dataset

In [ ]:
# Load the SMS Spam Collection dataset
# Download from: https://archive.ics.uci.edu/ml/datasets/sms+spam+collection
# OR use the inline sample below for testing

try:
    df = pd.read_csv('spam.csv', encoding='latin-1')
    df = df[['v1', 'v2']]
    df.columns = ['label', 'message']
    print(f'✓ Dataset loaded from file: {len(df)} messages')
except FileNotFoundError:
    # Fallback: use a representative sample dataset
    data = {
        'label': [
            'spam','spam','spam','spam','spam','spam','spam','spam','spam','spam',
            'ham','ham','ham','ham','ham','ham','ham','ham','ham','ham',
            'spam','spam','spam','ham','ham','ham','spam','ham','spam','ham'
        ],
        'message': [
            'FREE entry in 2 a weekly competition to win FA Cup final tickets!',
            'URGENT! You have won a 1 week FREE membership in our prize draw!',
            'Congratulations! You have been selected for a cash prize of $1000!',
            'Click here to claim your free iPhone now! Limited time offer!',
            'You have won a lottery prize. Call now to claim your winnings.',
            'FREE ringtone! Reply YES to get your free ringtone now!',
            'WINNER!! As a valued network customer you have been selected to receive a prize.',
            'Earn $500 per day working from home. No experience needed. Apply now!',
            'Your account has been suspended. Click here to verify your details.',
            'Special offer! 80% off on all products today only. Buy now!',
            'Hey, are you free tonight? Want to grab some dinner?',
            'Can you please send me the homework notes from class today?',
            'I will be late to the meeting, please start without me.',
            'Happy birthday! Hope you have a wonderful day.',
            'The project deadline has been extended to next Friday.',
            'Thanks for helping me yesterday, really appreciate it!',
            'See you at the library at 5pm for the study session.',
            'Mom called, she wants you to call her back when free.',
            'The doctor appointment is confirmed for tomorrow at 10am.',
            'Your package has been shipped. Expected delivery in 3 days.',
            'Win a brand new car! Enter our sweepstakes now!',
            'Cheap loans available! Bad credit ok! Apply instantly online!',
            'You are pre-approved for a credit card with zero interest!',
            'Lunch today? The cafeteria has your favourite pasta!',
            'Can you review my report before submitting it tomorrow?',
            'We are running low on groceries, can you pick some up?',
            'Get paid to take surveys! Earn $50 per hour from home!',
            'Reminder: Your dentist appointment is on Thursday.',
            'Reply to this message and win a free holiday vacation!',
            'Do not forget to bring your ID card to the exam hall.'
        ]
    }
    df = pd.DataFrame(data)
    print(f'✓ Sample dataset created: {len(df)} messages')

print(f"\nDataset Shape: {df.shape}")
print(f"Label Distribution:\n{df['label'].value_counts()}")
df.head()

## Step 3: Text Cleaning & Preprocessing

In [ ]:
def clean_text(text):
    """Clean and normalize text for embedding."""
    if isinstance(text, str):
        text = re.sub(r'[^A-Za-z\s]', '', text)  # Remove non-alphabetic chars
        text = text.lower().strip()               # Lowercase
        text = re.sub(r'\s+', ' ', text)          # Remove extra spaces
    else:
        text = ''
    return text

# Apply cleaning
df['cleaned_message'] = df['message'].apply(clean_text)
df = df[df['cleaned_message'].str.strip() != '']  # Remove empty rows

print('✓ Text preprocessing complete!')
print(f'\nSample cleaned messages:')
for i, row in df.head(3).iterrows():
    print(f"  [{row['label'].upper()}] {row['cleaned_message'][:80]}")

## Step 4: Sentence Embedding using Hugging Face MiniLM

In [ ]:
# Load the MiniLM model (same as Hadith Bot)
print('Loading MiniLM model from Hugging Face...')
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
print('✓ Model loaded successfully!')

# Generate embeddings for all messages
print('\nGenerating embeddings for all messages...')
embeddings = model.encode(df['cleaned_message'].tolist(), show_progress_bar=True)

print(f'\n✓ Embeddings generated!')
print(f'  - Number of messages: {embeddings.shape[0]}')
print(f'  - Embedding dimension: {embeddings.shape[1]}')

## Step 5: Vector Indexing using FAISS

In [ ]:
# Build FAISS index (Euclidean distance / L2)
d = embeddings.shape[1]  # Embedding dimension
index = faiss.IndexFlatL2(d)
index.add(embeddings.astype(np.float32))

# Save index for reuse in Flask app
faiss.write_index(index, 'spam_faiss.index')

# Save dataframe too
df.to_csv('spam_data_processed.csv', index=False)

print('✓ FAISS index built and saved!')
print(f'  - Total vectors indexed: {index.ntotal}')
print(f'  - Index file saved: spam_faiss.index')
print(f'  - Data file saved: spam_data_processed.csv')

## Step 6: Search / Retrieval Function

In [ ]:
def retrieve_similar_messages(query, model, index, df, k=5):
    """
    Given a query, find the top-k most similar messages from the dataset.
    Returns their labels (spam/ham) and similarity distances.
    """
    cleaned_query = clean_text(query)
    query_embedding = model.encode([cleaned_query]).astype(np.float32)
    
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for i in range(k):
        idx = indices[0][i]
        results.append({
            'rank': i + 1,
            'message': df['message'].iloc[idx],
            'label': df['label'].iloc[idx],
            'distance': round(float(distances[0][i]), 4)
        })
    return results


def predict_spam(query, model, index, df, k=5):
    """
    Predict if a message is spam or ham based on majority vote
    of the top-k most similar messages.
    """
    results = retrieve_similar_messages(query, model, index, df, k)
    labels = [r['label'] for r in results]
    spam_count = labels.count('spam')
    ham_count = labels.count('ham')
    
    prediction = 'spam' if spam_count > ham_count else 'ham'
    confidence = max(spam_count, ham_count) / k * 100
    
    return prediction, confidence, results

print('✓ Search functions defined!')

## Step 7: Test with Sample Queries

In [ ]:
test_queries = [
    "You have won a free prize, claim it now!",
    "Can we meet for lunch tomorrow?",
    "Earn money fast from home, no experience required!",
    "Please send me the homework assignment"
]

print('=' * 70)
print('SPAM QnA BOT - SIMILARITY SEARCH RESULTS')
print('=' * 70)

for query in test_queries:
    prediction, confidence, results = predict_spam(query, model, index, df)
    
    emoji = '🚫' if prediction == 'spam' else '✅'
    print(f'\n📩 Query: "{query}"')
    print(f'{emoji} Prediction: {prediction.upper()} ({confidence:.0f}% confidence)')
    print(f'\n  Top 5 Similar Messages:')
    for r in results:
        label_icon = '🚫' if r['label'] == 'spam' else '✅'
        print(f"  {r['rank']}. {label_icon} [{r['label'].upper()}] {r['message'][:60]}... (dist: {r['distance']})")
    print('-' * 70)

## Step 8: Launch Flask Web App
Run the next cell to start the Flask server, then open **http://127.0.0.1:5000** in your browser.

In [ ]:
# ============================================================
# FLASK WEB APPLICATION — Spam QnA Bot UI
# ============================================================
from flask import Flask, request, jsonify
import threading

# Load saved index and data (in case Flask runs separately)
loaded_index = faiss.read_index('spam_faiss.index')
loaded_df = pd.read_csv('spam_data_processed.csv')
flask_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

app = Flask(__name__)

HTML_PAGE = '''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Spam Detection QnA Bot</title>
  <link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600&display=swap" rel="stylesheet">
  <style>
    :root {
      --bg: #0d0d0d;
      --surface: #161616;
      --border: #2a2a2a;
      --accent: #ff4545;
      --green: #22c55e;
      --text: #e8e8e8;
      --muted: #666;
    }
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      background: var(--bg);
      color: var(--text);
      font-family: 'DM Sans', sans-serif;
      min-height: 100vh;
      padding: 2rem;
    }
    header {
      text-align: center;
      margin-bottom: 3rem;
      padding-top: 1rem;
    }
    header h1 {
      font-family: 'Space Mono', monospace;
      font-size: 2.2rem;
      letter-spacing: -1px;
    }
    header h1 span { color: var(--accent); }
    header p { color: var(--muted); margin-top: 0.5rem; font-size: 0.95rem; }
    .container { max-width: 780px; margin: 0 auto; }
    .input-box {
      background: var(--surface);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 1.5rem;
      margin-bottom: 1.5rem;
    }
    .input-box label {
      font-family: 'Space Mono', monospace;
      font-size: 0.75rem;
      color: var(--muted);
      text-transform: uppercase;
      letter-spacing: 2px;
      display: block;
      margin-bottom: 0.75rem;
    }
    textarea {
      width: 100%;
      background: #0d0d0d;
      border: 1px solid var(--border);
      border-radius: 8px;
      color: var(--text);
      font-family: 'DM Sans', sans-serif;
      font-size: 1rem;
      padding: 0.9rem 1rem;
      resize: none;
      outline: none;
      transition: border-color 0.2s;
    }
    textarea:focus { border-color: var(--accent); }
    button {
      width: 100%;
      margin-top: 1rem;
      padding: 0.9rem;
      background: var(--accent);
      color: white;
      font-family: 'Space Mono', monospace;
      font-size: 0.9rem;
      font-weight: 700;
      border: none;
      border-radius: 8px;
      cursor: pointer;
      letter-spacing: 1px;
      transition: opacity 0.2s, transform 0.1s;
    }
    button:hover { opacity: 0.85; }
    button:active { transform: scale(0.99); }
    button:disabled { opacity: 0.4; cursor: not-allowed; }
    #result { display: none; }
    .verdict {
      border-radius: 12px;
      padding: 1.5rem 2rem;
      margin-bottom: 1.5rem;
      text-align: center;
      border: 1px solid;
    }
    .verdict.spam { border-color: var(--accent); background: rgba(255,69,69,0.07); }
    .verdict.ham { border-color: var(--green); background: rgba(34,197,94,0.07); }
    .verdict-label {
      font-family: 'Space Mono', monospace;
      font-size: 2rem;
      font-weight: 700;
      letter-spacing: 3px;
    }
    .verdict.spam .verdict-label { color: var(--accent); }
    .verdict.ham .verdict-label { color: var(--green); }
    .verdict-conf {
      font-size: 0.9rem;
      color: var(--muted);
      margin-top: 0.4rem;
      font-family: 'Space Mono', monospace;
    }
    .section-title {
      font-family: 'Space Mono', monospace;
      font-size: 0.72rem;
      color: var(--muted);
      text-transform: uppercase;
      letter-spacing: 2px;
      margin-bottom: 0.75rem;
    }
    .results-list { display: flex; flex-direction: column; gap: 0.6rem; }
    .result-item {
      background: var(--surface);
      border: 1px solid var(--border);
      border-radius: 8px;
      padding: 0.85rem 1rem;
      display: flex;
      align-items: flex-start;
      gap: 0.75rem;
    }
    .rank {
      font-family: 'Space Mono', monospace;
      font-size: 0.75rem;
      color: var(--muted);
      min-width: 24px;
      padding-top: 2px;
    }
    .msg-text { font-size: 0.9rem; line-height: 1.5; flex: 1; }
    .label-badge {
      font-family: 'Space Mono', monospace;
      font-size: 0.7rem;
      font-weight: 700;
      padding: 0.2rem 0.55rem;
      border-radius: 4px;
      text-transform: uppercase;
      letter-spacing: 1px;
      min-width: 48px;
      text-align: center;
    }
    .label-badge.spam { background: rgba(255,69,69,0.15); color: var(--accent); }
    .label-badge.ham { background: rgba(34,197,94,0.15); color: var(--green); }
    .dist { font-family: 'Space Mono', monospace; font-size: 0.7rem; color: var(--muted); padding-top: 3px; }
    .loading { text-align: center; padding: 2rem; color: var(--muted); font-family: 'Space Mono', monospace; font-size: 0.85rem; letter-spacing: 2px; }
    .examples { margin-bottom: 1.5rem; }
    .examples p { font-size: 0.82rem; color: var(--muted); margin-bottom: 0.5rem; font-family: 'Space Mono', monospace; text-transform: uppercase; letter-spacing: 1.5px; }
    .chips { display: flex; flex-wrap: wrap; gap: 0.5rem; }
    .chip {
      background: var(--surface);
      border: 1px solid var(--border);
      padding: 0.35rem 0.75rem;
      border-radius: 999px;
      font-size: 0.8rem;
      cursor: pointer;
      transition: border-color 0.2s;
    }
    .chip:hover { border-color: var(--accent); }
  </style>
</head>
<body>
  <div class="container">
    <header>
      <h1>🛡️ <span>Spam</span> Detection Bot</h1>
      <p>MiniLM + FAISS Semantic Similarity Search — Lab 12</p>
    </header>

    <div class="examples">
      <p>Try an example</p>
      <div class="chips">
        <span class="chip" onclick="useExample(this)">You won a free prize!</span>
        <span class="chip" onclick="useExample(this)">Can we meet for lunch?</span>
        <span class="chip" onclick="useExample(this)">Earn money fast from home!</span>
        <span class="chip" onclick="useExample(this)">Please send me the notes</span>
        <span class="chip" onclick="useExample(this)">Claim your free iPhone now</span>
      </div>
    </div>

    <div class="input-box">
      <label>Enter a Message to Analyse</label>
      <textarea id="query" rows="3" placeholder="Type or paste any SMS/email message here..."></textarea>
      <button id="btn" onclick="analyze()">ANALYZE MESSAGE →</button>
    </div>

    <div id="result"></div>
  </div>

  <script>
    function useExample(el) {
      document.getElementById('query').value = el.textContent;
    }

    async function analyze() {
      const query = document.getElementById('query').value.trim();
      if (!query) return;

      const btn = document.getElementById('btn');
      const resultDiv = document.getElementById('result');
      btn.disabled = true;
      resultDiv.style.display = 'block';
      resultDiv.innerHTML = '<div class="loading">ANALYZING...</div>';

      try {
        const res = await fetch('/predict', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ query })
        });
        const data = await res.json();

        const isSpam = data.prediction === 'spam';
        const icon = isSpam ? '🚫' : '✅';

        let html = `
          <div class="verdict ${data.prediction}">
            <div class="verdict-label">${icon} ${data.prediction.toUpperCase()}</div>
            <div class="verdict-conf">${data.confidence}% confidence based on top-5 matches</div>
          </div>
          <p class="section-title">Top 5 Similar Messages from Dataset</p>
          <div class="results-list">`;

        data.results.forEach(r => {
          html += `
            <div class="result-item">
              <span class="rank">#${r.rank}</span>
              <span class="msg-text">${r.message}</span>
              <span class="label-badge ${r.label}">${r.label}</span>
              <span class="dist">${r.distance}</span>
            </div>`;
        });

        html += '</div>';
        resultDiv.innerHTML = html;
      } catch (e) {
        resultDiv.innerHTML = '<div class="loading">Error — is Flask running?</div>';
      }
      btn.disabled = false;
    }
  </script>
</body>
</html>
'''

@app.route('/')
def home():
    return HTML_PAGE

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    query = data.get('query', '')
    
    prediction, confidence, results = predict_spam(
        query, flask_model, loaded_index, loaded_df, k=5
    )
    
    return jsonify({
        'prediction': prediction,
        'confidence': round(confidence, 1),
        'results': results
    })

# Launch Flask in a background thread (works inside Jupyter)
def run_flask():
    app.run(port=5000, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

print('✓ Flask server started!')
print('🌐 Open your browser at: http://127.0.0.1:5000')